## Cleaning Raw Data 
The purpose of this code is to clean the raw data set so that it can be used for the SIR modeling in R. 

## Data Source 
Historical data records were sourced from Lazzari, G., Colavizza, G., Bortoluzzi, F., Drago, D., Erboso, A., Zugno, F., ... & Salathé, M. (2020). A digital reconstruction of the 1630–1631 large plague outbreak in Venice. Scientific reports, 10(1), 1-7.

Importing the Primary modules used: 

In [151]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline

 ## Load in Data 

In [152]:
year_one_data = pd.read_csv('1630_Venice_Plague_Data.csv')
year_one_data.head()


,Date,Age,Sex,Cause of Death,Days Afflicted,Type of death
0,1/3/1630,22,F,Fever From Plague,60,Plague Death
1,1/4/1630,25,F,Fever From Plague,60,Plague Death
2,1/4/1630,14,M,Fever From Plague,60,Plague Death
3,1/8/1630,14,M,Fever From Plague,60,Plague Death
4,1/8/1630,5,M,Fever From Plague,60,Plague Death


In [153]:
year_two_data = pd.read_csv('1631_Venice_Plague_Data.csv')
year_two_data.head()

,Date,Age,Sex,Cause of Death,Afflicted Time (Days,Type of Death,Unnamed: 6
0,1/1/31,60,M,Coal,5,Non-Plague Death,NaN
1,1/3/31,73,F,apoplexy,365,Non-Plague Death,NaN
2,1/4/31,5,M,Infection,3,Non-Plague Death,NaN
3,1/4/31,108,F,Plague and Old age,1095,Plague Death,NaN
4,1/4/31,50,M,Plague,10,Plague Death,NaN


## Clean Year One Data 

Calculate Summary Statistics: 

In [154]:
print(year_one_data.describe())

              Date  Age  Sex      Cause of Death Days Afflicted   \
count          925  924  924                 925             919   
unique         211   78    3                  76              44   
top     10/31/1630    0    F  Most Likely Plague               4   
freq            22   72  477                 157             128   

       Type of death   
count             925  
unique              2  
top      Plague Death  
freq              737  


 Determine Data Types: 

In [155]:
print(year_one_data.dtypes)

Date               object
Age                object
Sex                object
Cause of Death     object
Days Afflicted     object
Type of death      object
dtype: object


Change Date Type:

In [156]:
year_one_data['Date'] = year_one_data['Date'].str.rsplit('/', n=1).str[0]

year_one_data['Date'] = pd.to_datetime(
    year_one_data['Date'],
    format='%m/%d'
)

In [157]:
print(year_one_data['Date'].head())

0   1900-01-03
1   1900-01-04
2   1900-01-04
3   1900-01-08
4   1900-01-08
Name: Date, dtype: datetime64[ns]


In [158]:
print(year_one_data['Date'].dtypes)

datetime64[ns]


Inspect Data for Null Values:

In [159]:
print(year_one_data.isna().sum())

Date               0
Age                1
Sex                1
Cause of Death     0
Days Afflicted     6
Type of death      0
dtype: int64


In [160]:
year_one_data.dropna(inplace=True)

Create Time Index:

In [161]:
year_one_data['Days Since Start'] = year_one_data['Date'] - year_one_data['Date'].min()

In [162]:
year_one_data.head()

,Date,Age,Sex,Cause of Death,Days Afflicted,Type of death,Days Since Start
0,1900-01-03,22,F,Fever From Plague,60,Plague Death,0 days
1,1900-01-04,25,F,Fever From Plague,60,Plague Death,1 days
2,1900-01-04,14,M,Fever From Plague,60,Plague Death,1 days
3,1900-01-08,14,M,Fever From Plague,60,Plague Death,5 days
4,1900-01-08,5,M,Fever From Plague,60,Plague Death,5 days


Plague Flag:

In [163]:
year_one_data['Plague Flag'] = (
    year_one_data['Type of death '] == 'Plague Death').astype(int)

In [164]:
year_one_data.head()

,Date,Age,Sex,Cause of Death,Days Afflicted,Type of death,Days Since Start,Plague Flag
0,1900-01-03,22,F,Fever From Plague,60,Plague Death,0 days,1
1,1900-01-04,25,F,Fever From Plague,60,Plague Death,1 days,1
2,1900-01-04,14,M,Fever From Plague,60,Plague Death,1 days,1
3,1900-01-08,14,M,Fever From Plague,60,Plague Death,5 days,1
4,1900-01-08,5,M,Fever From Plague,60,Plague Death,5 days,1


Create Age Groups:

In [167]:
year_one_data['Age'] = year_one_data['Age'].replace('[Unkown]', '0', regex = True) #Assigning "unknown' and blank ages to 0 so that age can be converted to int

In [168]:
year_one_data['Age'] = year_one_data['Age'].replace('["      "]', '0', regex = True)

In [169]:
year_one_data['Age'] = pd.to_numeric(year_one_data['Age'])

In [170]:
year_one_data['Age Group'] = np.select(
    [
        year_one_data['Age'] < 1,
        year_one_data['Age'] < 13,
        year_one_data['Age'] < 20,
        year_one_data['Age'] < 40,
        year_one_data['Age'] < 60
    ],
    [
        'Infant',
        'Child',
        'Teen',
        'Young Adult',
        'Middle Age'
    ],
    default='Older Adult'
)

In [171]:
year_one_data.head()

,Date,Age,Sex,Cause of Death,Days Afflicted,Type of death,Days Since Start,Plague Flag,Age Group
0,1900-01-03,22,F,Fever From Plague,60,Plague Death,0 days,1,Young Adult
1,1900-01-04,25,F,Fever From Plague,60,Plague Death,1 days,1,Young Adult
2,1900-01-04,14,M,Fever From Plague,60,Plague Death,1 days,1,Teen
3,1900-01-08,14,M,Fever From Plague,60,Plague Death,5 days,1,Teen
4,1900-01-08,5,M,Fever From Plague,60,Plague Death,5 days,1,Child


Write Cleaned CSV File:

In [172]:
year_one_data.to_csv('cleaned_plague_data_1630.csv', index=False)

# Clean Year Two Data 

Calculate Summary Statistics:

In [173]:
print(year_two_data.describe())

          Date  Age  Sex  Cause of Death Afflicted Time (Days Type of Death  \
count       859  848  859            857                  850           859   
unique      271   80    2             66                   39             2   
top     6/13/31    0    F         Plague                    4  Plague Death   
freq         10   40  440            284                  175           430   

             Unnamed: 6  
count                 3  
unique                2  
top     età non segnata  
freq                  2  


Drop 'unnamed' Column:

In [174]:
year_two_data.drop(columns=['Unnamed: 6'], inplace=True)

In [175]:
year_two_data.head()

,Date,Age,Sex,Cause of Death,Afflicted Time (Days,Type of Death
0,1/1/31,60,M,Coal,5,Non-Plague Death
1,1/3/31,73,F,apoplexy,365,Non-Plague Death
2,1/4/31,5,M,Infection,3,Non-Plague Death
3,1/4/31,108,F,Plague and Old age,1095,Plague Death
4,1/4/31,50,M,Plague,10,Plague Death


Determine Data Types: 

In [176]:
year_two_data.dtypes

Date                    object
Age                     object
Sex                     object
Cause of Death          object
Afflicted Time (Days    object
Type of Death           object
dtype: object

Change Date to Correct Type: 

In [177]:
year_two_data.rename(
    columns={
        'Date ': 'Date',
        'Age ': 'Age'
    },
    inplace=True
)

In [178]:
year_two_data['Date'] = year_two_data['Date'].str.rsplit('/', n=1).str[0]
year_two_data['Date'] = pd.to_datetime(
    year_two_data['Date'],
    format='%m/%d'
)

In [179]:
print(year_two_data['Date'].dtypes)

datetime64[ns]


Inspect For Null Values:

In [180]:
print(year_two_data.isna().sum())

Date                     0
Age                     11
Sex                      0
Cause of Death           2
Afflicted Time (Days     9
Type of Death            0
dtype: int64


In [181]:
year_two_data.dropna(inplace=True)

Create Time Index:  

In [182]:
year_two_data['Days Since Start'] = year_two_data['Date'] - year_two_data['Date'].min()

In [183]:
year_two_data.head()

,Date,Age,Sex,Cause of Death,Afflicted Time (Days,Type of Death,Days Since Start
0,1900-01-01,60,M,Coal,5,Non-Plague Death,0 days
1,1900-01-03,73,F,apoplexy,365,Non-Plague Death,2 days
2,1900-01-04,5,M,Infection,3,Non-Plague Death,3 days
3,1900-01-04,108,F,Plague and Old age,1095,Plague Death,3 days
4,1900-01-04,50,M,Plague,10,Plague Death,3 days


Create Plague Flag: 

In [184]:
year_two_data['Plague Flag'] = (
    year_two_data['Type of Death'] == 'Plague Death').astype(int)

In [185]:
year_two_data.head()

,Date,Age,Sex,Cause of Death,Afflicted Time (Days,Type of Death,Days Since Start,Plague Flag
0,1900-01-01,60,M,Coal,5,Non-Plague Death,0 days,0
1,1900-01-03,73,F,apoplexy,365,Non-Plague Death,2 days,0
2,1900-01-04,5,M,Infection,3,Non-Plague Death,3 days,0
3,1900-01-04,108,F,Plague and Old age,1095,Plague Death,3 days,1
4,1900-01-04,50,M,Plague,10,Plague Death,3 days,1


Create Age Groups: 

In [186]:
year_two_data['Age'] = year_two_data['Age'].replace('[Unkown]', '0', regex = True)

In [187]:
year_two_data['Age'] = pd.to_numeric(year_two_data['Age'])

In [188]:
year_two_data['Age Group'] = np.select(
    [
        year_two_data['Age'] < 1,
        year_two_data['Age'] < 13,
        year_two_data['Age'] < 20,
        year_two_data['Age'] < 40,
        year_two_data['Age'] < 60
    ],
    [
        'Infant',
        'Child',
        'Teen',
        'Young Adult',
        'Middle Age'
    ],
    default='Older Adult'
)

In [189]:
year_two_data.head()

,Date,Age,Sex,Cause of Death,Afflicted Time (Days,Type of Death,Days Since Start,Plague Flag,Age Group
0,1900-01-01,60,M,Coal,5,Non-Plague Death,0 days,0,Older Adult
1,1900-01-03,73,F,apoplexy,365,Non-Plague Death,2 days,0,Older Adult
2,1900-01-04,5,M,Infection,3,Non-Plague Death,3 days,0,Child
3,1900-01-04,108,F,Plague and Old age,1095,Plague Death,3 days,1,Older Adult
4,1900-01-04,50,M,Plague,10,Plague Death,3 days,1,Middle Age


Write Cleaned CSV file:

In [190]:
year_two_data.to_csv('cleaned_plague_data_1631.csv', index=False)